In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# automatically reload modules when they have changed
%load_ext autoreload
%autoreload 2

In [ ]:
from track2p.ops.default import DefaultTrackOps

from track2p.io.s2p_loaders import load_all_imgs, check_nplanes
from track2p.io.savers import save_track_ops

from track2p.register.loop import run_reg_loop, reg_all_ds_all_roi
from track2p.register.utils import get_all_ds_img_for_reg, get_all_ref_nonref_inters

from track2p.plot.progress import plot_all_planes
from track2p.plot.output import plot_reg_img_output, plot_roi_reg_output


### Start of algo

In [ ]:
# 1) Define parameters (TODO: GUI or CLI)
track_ops = DefaultTrackOps()

# 2) Load data
check_nplanes(track_ops)
all_ds_avg_ch1, all_ds_avg_ch2 = load_all_imgs(track_ops)

# 3) Plot available planes for registration (TODO: the user needs to make sure their recordings can be matched, i. e. we can spot the same cells by eye in the different recordings, TODO: allow user to choose interactiverly which channel to use?)
plot_all_planes(all_ds_avg_ch1, track_ops)
if track_ops.nchannels==2:
    plot_all_planes(all_ds_avg_ch2, track_ops)

In [ ]:
# 2) do the actual registration based on chosen channel
all_ds_ref_img, all_ds_mov_img = get_all_ds_img_for_reg(all_ds_avg_ch1, all_ds_avg_ch2, track_ops)
all_ds_mov_img_reg, all_ds_reg_params = run_reg_loop(all_ds_ref_img, all_ds_mov_img, track_ops) # TODO: save basic parameters for each registration as feedback (e. g. ammoung of shift, rotation, etc.) for later plotting
save_track_ops(track_ops)

### Visualising image output

In [ ]:
plot_reg_img_output(track_ops)

### Applying transform to all ROIs and visualising

In [ ]:
all_ds_all_roi_array_ref, all_ds_all_roi_array_mov, all_ds_all_roi_array_reg, all_ds_roi_counter = reg_all_ds_all_roi(all_ds_reg_params, track_ops)

In [ ]:
# compute intersection 
all_ds_ref_reg_inters = get_all_ref_nonref_inters(all_ds_all_roi_array_ref, all_ds_all_roi_array_reg, track_ops)
all_ds_ref_mov_inters = get_all_ref_nonref_inters(all_ds_all_roi_array_ref, all_ds_all_roi_array_mov, track_ops)

track_ops.all_ds_ref_mov_inters = all_ds_ref_mov_inters
track_ops.all_ds_ref_reg_inters = all_ds_ref_reg_inters

In [ ]:
# free up some memory
del all_ds_all_roi_array_ref, all_ds_all_roi_array_mov, all_ds_all_roi_array_reg

In [ ]:
# this line is very memory-intensive because of the ROIS (TODO: maybe instead of contours just plot RGB) (or somehow generate RGB image of contours (in the part before))
if track_ops.show_roi_reg_output:
    # TODO: for each condition generate the RGB image of contorus (so its more memory efficient)
    plot_roi_reg_output(track_ops)

In [ ]:
# TODO: for each ROI in the ref recording compute centroid distances (and maybe other metrics) with all ROIs in the reg recording and choose the best match

In [ ]:
# TODO: after matching plot the statistics of the neurons (how many are left, how many are new, how many are lost, etc.)

In [ ]:
# TODO: saving of the track_reg object